In [ ]:
!pip install comet_ml
import comet_ml
from comet_ml import Experiment
from comet_ml.integration.pytorch import log_model
!pip install lightning
import lightning
from lightning.fabric import Fabric
import pandas as pd
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader
import re
import os
import lzma
from tqdm import tqdm
import mmap
import random
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.7/586.7 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 26.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 kB 31.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.9/137.9 kB 19.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.3/54.3 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.7/514.7 kB 35.3 MB/s eta 0:00:00
  Attempting uninstall: websocket-client
    Found existing installation: websocket-client 1.6.4
    Uninstalling websocket-client-1.6.4:
      Successfully uninstalled websocket-client-1.6.4
  Attempting uninstall: python-box
    Found existing installation: python-box 7.1.1
    Uninstalling python-box-7.1.1:
      Successfully uninstalled python-box-7.1.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.9 MB/s eta

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
fabric = Fabric(accelerator="cuda", devices=1, precision="16-mixed")
device = fabric.device

INFO: Using 16-bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16-bit Automatic Mixed Precision (AMP)


# Loading Jokes dataset

In [ ]:
# Jokes dataset
!git clone https://github.com/taivop/joke-dataset.git
!git clone https://github.com/amoudgl/short-jokes-dataset.git
!git clone https://huggingface.co/datasets/shuttie/dadjokes

Cloning into 'joke-dataset'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 44 (delta 12), reused 10 (delta 10), pack-reused 30
Receiving objects: 100% (44/44), 32.38 MiB | 16.42 MiB/s, done.
Resolving deltas: 100% (21/21), done.
Cloning into 'short-jokes-dataset'...
remote: Enumerating objects: 49, done.
remote: Total 49 (delta 0), reused 0 (delta 0), pack-reused 49
Receiving objects: 100% (49/49), 34.56 MiB | 15.15 MiB/s, done.
Resolving deltas: 100% (20/20), done.
Cloning into 'dadjokes'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 12 (delta 2), reused 0 (delta 0), pack-reused 4
Unpacking objects: 100% (12/12), 2.80 MiB | 2.18 MiB/s, done.


## joke-dataset

In [ ]:
# joke-dataset 200k
reddit_path = "/content/joke-dataset/reddit_jokes.json"
stupid_path = "/content/joke-dataset/stupidstuff.json"
wocka_path = "/content/joke-dataset/wocka.json"

reddit_df = pd.read_json(reddit_path)
stupid_df = pd.read_json(stupid_path)
wocka_df = pd.read_json(wocka_path)

# extract features
reddit_data = reddit_df['title'] + ' ' + reddit_df['body']
stupid_data = stupid_df['body']
wocka_data = wocka_df['body']

dataframes = [reddit_data, stupid_data, wocka_data]
jokes_df = pd.concat(dataframes, ignore_index=True)

In [ ]:
jokes_df.head()

0    I hate how you cant even say black paint anymo...
1    What's the difference between a Jew in Nazi Ge...
2    I recently went to America.... ...and being th...
3    Brian raises his hand and says, “He’s in Heave...
4    You hear about the University book store worke...
dtype: object

## dad jokes

In [ ]:
# dadjokes 53.4k
train_path = "/content/dadjokes/train.csv"
test_path = "/content/dadjokes/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_data = train_df['question'] + ' ' + train_df['response']
test_data = test_df['question'] + ' ' + test_df['response']

dataframes = [train_data, test_data]
dadjokes_df = pd.concat(dataframes)

In [ ]:
dadjokes_df.head()

0    I asked my priest how he gets holy water He sa...
1    Life Hack: If you play My Chemical Romance lou...
2    OMG. SISTERS. JAMES. CHARLES. IS. DOING. A GIV...
3    Why did Mr.  Potato Head get pulled over He wa...
4    On zombie cravings.  My kids and i had some fu...
dtype: object

## short jokes

In [ ]:
# short_jokes_dataset 230k
short_jokes_path = "/content/short-jokes-dataset/shortjokes.csv"

short_jokes_df = pd.read_csv(short_jokes_path)
short_jokes_df = short_jokes_df['Joke']

In [ ]:
short_jokes_df.head()

0    [me narrating a documentary about narrators] "...
1    Telling my daughter garlic is good for you. Go...
2    I've been going through a really rough period ...
3    If I could have dinner with anyone, dead or al...
4       Two guys walk into a bar. The third guy ducks.
Name: Joke, dtype: object

# Data Preprocessing

In [ ]:
# Redefining the clean_text function
def clean_text(text):
  # convert into string
  text = str(text)
  # Lowercase the text
  text = text.lower()
  # Remove words that contain numbers
  text = re.sub(r'\w*\d\w*', '', text)
  # remove digits longer than 5
  text = re.sub(r'\d{5,}', '', text)
  # Preserve meaningful punctuation and numbers
  text = re.sub(r"[^a-z0-9.,!?;\s']", '', text)
  # Remove words with characters other than letters in them
  text = re.sub(r'\s\w+[.,!?;]\w+\s', ' ', text)
  words = text.split()
  text = ' '.join(words)
  # Truncate text to start after the first period and end at the last period
  start = text.find('.') + 1
  end = text.rfind('.')
  if start != 0 and end != -1 and end > start:
      text = text[start:end].strip()
  text = re.sub(r'\,+[,\s]+[^\w]', ', ', text) # Remove occurences such as , ,
  text = re.sub(r'\.+[\.\s]+[^\w]', '. ', text) # Remove occurences such as . .
  text = re.sub(r'\s+\.', '.' , text) # If there is space before '.' remove it
  text = re.sub(r'\s+[,]', ',' , text) # If there is space before ',' remove it
  text = re.sub(r'\s+[!]', '!' , text) # If there is space before '!' remove it
  text = re.sub(r'\s+[?]', '?' , text) # If there is space before '?' remove it
  # Remove unnecessary whitespaces and handle line breaks
  text = text.strip()
  text = re.sub(r'\s+', ' ', text)
  # Remove repeating commas and periods
  text = re.sub(r'[.]+', '.', text)
  text = re.sub(r'[,]+', ',', text)
  text = re.sub(r'\.+[,]+', ',', text)
  return text

def is_valid_line(line):
  if line:

    if len(line) < 2:
      return False

    if not re.search('[a-z]', line):
      return False

    return True

  return False

In [ ]:
# combine 3 datafraes
JOKES_df = pd.concat([dadjokes_df, jokes_df, short_jokes_df], axis=0)

# remove duplicates
JOKES_df = JOKES_df.drop_duplicates()

In [ ]:
JOKES_df = JOKES_df.apply(clean_text)

In [ ]:
JOKES_df

0         I asked my priest how he gets holy water He sa...
1         Life Hack: If you play My Chemical Romance lou...
2         OMG. SISTERS. JAMES. CHARLES. IS. DOING. A GIV...
3         Why did Mr.  Potato Head get pulled over He wa...
4         On zombie cravings.  My kids and i had some fu...
                                ...                        
231650    is this already a joke? Why don't pastry chefs...
231652                  The Spicy Sausage by Delia Katessen
231653    TIL That I Shouldn't have gone to law school, ...
231655    what do you call a play about victorian era me...
231656    Calculus should be taught in every high school...
Length: 402878, dtype: object

# Tokenize

In [ ]:
chars = ''
with open('/content/drive/MyDrive/Project/Minipile/mini_pile_train_vocab.txt', 'r') as f:
  chars = f.read()

In [ ]:
print(len(chars))
print(chars)

34

 !',.;?abcdefghijklmnopqrstuvwxyz


In [ ]:
stoi = { ch:i for i, ch in enumerate(chars)}
itos = { i:ch for i, ch in enumerate(chars)}

# encoder: string to int
encode = lambda s: [stoi[c] for c in s if c in stoi]

# decoder: int to string
decode = lambda l: ''.join([itos[i] for i in l if i in itos])

In [ ]:
# read the text file
%cd /content/drive/MyDrive/Project/Jokes_Datasets
with open('jokes.txt', 'r', encoding='utf-8') as file:
  data = torch.tensor(encode(file.read()), dtype=torch.long, device=device)
  n = int(0.9*len(data))
  train_data = data[:n]
  val_data = data[n:]

print(f"length of text: {len(data)}")

/content/drive/.shortcut-targets-by-id/1DUoPwqF9OmnYJmuAwgDAFNsyXvHQQS9l/Project/Jokes_Datasets
length of text: 51649977


# Model

In [ ]:
# single head
class Head(nn.Module):

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)

    wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)

    v = self.value(x)
    out = wei @ v
    return out

# multi-head
class MultiHeadAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embd, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out


class FeedForward(nn.Module):

  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4 * n_embd),
        nn.ReLU(),
        nn.Linear(4 * n_embd, n_embd),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)

class Block(nn.Module):

  def __init__(self, n_embd, n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x

In [ ]:
class GPTLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd)
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
    B, T = idx.shape

    tok_emb = self.token_embedding_table(idx)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device))
    x = tok_emb + pos_emb
    x = self.blocks(x)
    x = self.ln_f(x)
    logits = self.lm_head(x)

    if targets is None:
        loss = None
    else:
        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, loss = self(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx


# Batches

In [ ]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  losses = torch.zeros(eval_iters)
  for split in ['train', 'val']:
    for k in range(100):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

# Training

## Hyperparameters

In [ ]:
# hyperparameters
batch_size = 1024
block_size = 128
max_iters = 2000
eval_interval = 100
learning_rate = 1e-3
eval_iters = 2000
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.2
#device = torch.device("cuda")
vocab_size = len(chars)

hyperparameters = {
    'batch_size': batch_size,
    'block_size': block_size,
    'max_iters': max_iters,
    'eval_interval': eval_interval,
    'learning_rate': learning_rate,
    'eval_iters': eval_iters,
    'n_embd': n_embd,
    'n_head': n_head,
    'n_layer': n_layer,
    'dropout': dropout,
}

# Hyperparameter -> Loss Visualization Data
# Essentially for every run, record the hyperparams and
# Plot them compared to old (save old in file)
vis_data = {
    'batch_size': batch_size,
    'block_size': block_size,
    'max_iters': max_iters,
    'eval_interval': eval_interval,
    'learning_rate': learning_rate,
    'eval_iters': eval_iters,
    'n_embd': n_embd,
    'n_head': n_head,
    'n_layer': n_layer,
    'dropout': dropout,
}

vis_data_file_name = "visual_data_during_training.txt"
vis_post_data_file_name = "visual_data_post_training.txt"

In [ ]:
# model
model = GPTLanguageModel()
model.to(device)
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

# Visualization Data
vis_data_iterations = [] # Format: [[iteration, train_loss, val_loss]]

# optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
'''# load the model and optimizer

model_384_8_8_path = '/content/drive/MyDrive/Project/Saved model/model_384_8_8.pth'
model_384_16_16_path = '/content/drive/MyDrive/Project/Saved model/model_384_16_16.pth'
model_128_4_4_path = '/content/drive/MyDrive/Project/Saved model/model_128_4_4.pth'

checkpoint = torch.load(model_384_8_8_path)
checkpoint = torch.load(model_384_16_16_path)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
'''

In [ ]:
%cd /content/drive/
model_128_4_4_path = '/content/drive/MyDrive/Project/Saved model/model_128_4_4.pth'
checkpoint = torch.load(model_128_4_4_path)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [ ]:
fabric.launch()

experiment = Experiment(
  api_key="78u2AfbhkXeTChB3Kzb7FhOEY",
  project_name="JokeGPT",
  workspace="lzh0212"
)

experiment.log_parameters(hyperparameters)

for iter in range(max_iters):
  # if iter % eval_interval == 0 or iter == max_iters - 1:
  #   losses = estimate_loss()
  #   print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

  xb, yb = get_batch('train')
  logits, loss = model(xb, yb)

  if iter % eval_interval == 0 or iter == max_iters - 1:
    losses = estimate_loss()
    experiment.log_metric("train_loss", losses['train'], epoch=iter)
    experiment.log_metric("val_loss", losses['val'], epoch=iter)

  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

experiment.end()

In [ ]:
losses = estimate_loss()
print(f"train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

# Save the model

In [ ]:
def save(model, optimizer, hyperparameters, base_path='/content/drive/MyDrive/Project/Saved model/'):
    n_embd = hyperparameters['n_embd']
    n_head = hyperparameters['n_head']
    n_layer = hyperparameters['n_layer']

    filename = f"fine_tuned_model_{n_embd}_{n_head}_{n_layer}.pth"
    full_path = base_path + filename

    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'hyperparameters': hyperparameters
    }, full_path)

    print(f"Checkpoint saved to {full_path}")

In [ ]:
save(model, optimizer, hyperparameters)

# Result

In [ ]:
base_model_128_4_4_path = '/content/drive/MyDrive/Project/Saved model/model_128_4_4.pth'
fined_tuned_model_128_4_4_path = '/content/drive/MyDrive/Project/Saved model/fine_tuned_model_128_4_4.pth'

In [ ]:
model = GPTLanguageModel()
model.to(device)

path = fined_tuned_model_128_4_4_path
checkpoint = torch.load(path)
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [ ]:
prompt = 'university of california los angeles'
context = torch.tensor(encode(prompt), dtype=torch.long, device=device)
generated_chars = decode(model.generate(context.unsqueeze(0), max_new_tokens=100)[0].tolist())
print(generated_chars)

university of california los angeles. yo mama rubbes rantule poured for just about food and walking rabbits? hitting your mind if you ca


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


the worked hard work hard, lily, petexture at the park to place and caught and happened. she followed the before saw the spicy shoulder looked delicious!
endoftext

sam went to both a time, there was a little girl named whenever he got smaller and whispers. soon wonderfully his sound and hugged everywhere. one day, the old landscape for hours were very strong. they were happy and like to play with the pool. the bear was very angry. 
the little girl was getting his light. as the pieces went wrong
